# Токенайзер

In [262]:
from nltk.tokenize import TweetTokenizer

In [263]:
from string import punctuation

In [264]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [265]:
from nltk.corpus import stopwords

In [266]:
from sklearn.feature_extraction.text import CountVectorizer

In [267]:
def tokenizer(t):
  t = t.lower()
  tokeniz = TweetTokenizer()
  tokeny = tokeniz.tokenize(t)
  tokeny = [tok for tok in tokeny if (tok not in stopwords.words('russian')) and (tok not in punctuation)]
  return tokeny

In [268]:
from nltk.stem.snowball import SnowballStemmer

In [269]:
st = SnowballStemmer('russian')

In [270]:
tok = tokenizer('Попытка сделать хакатон по ТП')

In [271]:
final = [st.stem(x) for x in tok]
final

['попытк', 'сдела', 'хакатон', 'тп']

# Чтение файлов из папки для выявления ключевых слов, прогнанных через токенайзер


In [286]:
def search_for_sender(file):
  for line in file:
    if 'From' in line or 'from' in line or 'Ot kogo' in line or 'От кого' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          if '<' in mail and '>' in mail:
            return mail[1:-1]
          else:
            return mail
    else:
      return None

Извлечение из письма отправителя письма

In [273]:
def search_for_recipient(file):
  for line in file:
    if 'to' in line or 'Komu' in line or 'Кому' in line or 'To' in line:
      words = line.split()
      for mail in words:
        if '@' in mail:
          return mail
  else:
    return None

Извлечение из письма получателя письма

In [274]:
def search_text(file):
  text = ''
  for line in file:
    sender = search_for_sender(file)
    recipient = search_for_recipient(file)
    subject = search_subject(file)
    sender_valid = sender not in line
    recipient_valid = recipient is None or recipient not in line
    if sender_valid and recipient_valid:
      text += f'{line}'
  return text

Извлечение текста из письма, нужного для определения категории, в которую будет определено письмо


In [275]:
categories_weights = {
  'Спам': {
    'вложен': 0.03, 'выигра': 0.10, 'приз': 0.10, 'iphone': 0.09,
    'exclusive': 0.09, 'offer': 0.07, 'limited': 0.05, 'личност': 0.05,
    'заблокирова': 0.07, 'перейд': 0.07, 'ссылк': 0.05, 'password-reset': 0.05,
    'secure-login': 0.05, 'внешн': 0.05, 'парол': 0.01,
    'эксклюзивн': 0.04, 'акц': 0.03
  },
  'Важное': {
    'urgent': 0.12, 'срочн': 0.12, 'критическ': 0.12, 'массов': 0.09,
    'сбо': 0.12, 'работ': 0.08, 'остановл': 0.09, 'отвеча': 0.08,
    'недоступ': 0.08, 'утечк': 0.05, 'дан': 0.02, 'краж': 0.03
  },
  'Технические ошибки': {
    'ошибк': 0.07, 'error': 0.03, 'зависа': 0.08,
    'слома': 0.08, 'ddos': 0.05, 'ремонт': 0.05, 'перебо': 0.05,
    'интернет': 0.04, 'связ': 0.04, 'запуска': 0.04, 'обновлен': 0.04,
    'открыва': 0.03, 'компьютер': 0.03, 'ноутбук': 0.03, 'принтер': 0.03,
    'сканер': 0.03, 'outlook': 0.03, 'chrome': 0.03, 'excel': 0.03,
    'zoom': 0.03, 'мыш': 0.03,
    'гарнитур': 0.02, 'неисправн': 0.04,
    'подключ': 0.04, 'кнопк': 0.02, 'портал': 0.01
  },
  'Информация': {
    'созвон': 0.15, 'дайджест': 0.11, 'мониторинг': 0.11, 'healthcheck': 0.11,
    'дем': 0.11, 'отчет': 0.08, 'info': 0.08, 'планов': 0.06,
    'брифинг': 0.06, 'брейншторм': 0.05, 'обновлен': 0.04, 'встреч': 0.02,
    'работ': 0.02
  },
  'Документы': {
    'договор': 0.12, 'акт': 0.10, 'закрыва': 0.13, 'документ': 0.13,
    'согласован': 0.13, 'contract': 0.13, 'финальн': 0.10, 'верс': 0.03,
    'дан': 0.03, 'задан': 0.01, 'задание': 0.05, 'инструкц': 0.02, 'инструкция': 0.02
  },
  'Счета': {
    'счет': 0.25, 'оплат': 0.25, 'invoice': 0.15, 'реквизит': 0.10,
    'задолжен': 0.03, 'акт': 0.03, 'выписк': 0.03, 'кред': 0.03,
    'выплат': 0.03, 'бухгалтер': 0.03, 'страховк': 0.03, 'зарплат': 0.02,
    'прем': 0.01, 'компенсац': 0.01
  },
  'Подтверждение доступа к аккаунту': {
    'vpn': 0.12, 'gitlab': 0.12, 'confluence': 0.12, '1c': 0.12,
    'выда': 0.09, 'прав': 0.09, 'восстанов': 0.08, 'запрос': 0.07,
    'доступ': 0.07, 'логин': 0.05, 'аккаунт': 0.04, 'почт': 0.02,
    'парол': 0.01
  },
  'HR': {
    'отпуск': 0.10, 'больничн': 0.10, 'резюм': 0.09, 'оформлен': 0.08,
    'должност': 0.07, 'назначен': 0.07, 'повышен': 0.07, 'отпускн': 0.07,
    'опоздан': 0.07, 'нетрудоспособн': 0.07, 'график': 0.05, 'сотрудник': 0.04,
    'работ': 0.02, 'перевод': 0.05, 'отдел': 0.05
  }
}

Разбиение слов, прогнанных через токенайзер по весу слов. Сумма слов в категории дает 1.00, что дает равные шансы для попадания письма в каждую категорию, что дает более качественный результат.


In [276]:
def classify_email(text, weights_dict):
  z = []
  if not text or len(text) == 0:
    return "Спам"

  tokens = tokenizer(text)

  if len(tokens) == 0:
    return "Спам"

  scores = {category: 0.0 for category in weights_dict.keys()}

  for token in tokens:
    stemmed_token = st.stem(token)
    z.append(stemmed_token)
    for category, words_weights in weights_dict.items():
      if stemmed_token in words_weights:
        scores[category] += words_weights[stemmed_token]

  if max(scores.values()) <= 0.05:
    return "Не определено"

  return max(scores, key=scores.get)

In [295]:
folder_path = '/content/HAKATON/inbox/'
files = sorted(os.listdir(folder_path))

for file_name in files:
  if file_name.lower().endswith(('.jpeg', '.jpg', '.png')):
    print(f'{file_name} -> Картинки')
    continue

  elif file_name.endswith('.txt'):
    full_path = os.path.join(folder_path, file_name)

    with open(full_path, 'r', encoding='utf-8') as file:
      content = file.readlines()

    has_sender = False
    has_recipient = False
    clean_lines = []

    for line in content:
      line_lower = line.lower()

      if 'from:' in line_lower or 'от кого:' in line_lower or 'ot kogo:' in line_lower:
        if '@' in line:
          has_sender = True
          continue

      if 'to:' in line_lower or 'кому:' in line_lower or 'komu:' in line_lower:
        if '@' in line:
          has_recipient = True
          continue

      if 'date:' in line_lower or 'дата:' in line_lower:
        continue

      clean_lines.append(line)

    if not has_sender:
      print(f'{file_name} -> Ошибки')
      continue

    if not has_recipient:
      print(f'{file_name} -> Черновик')
      continue

    email_text = "".join(clean_lines).strip()

    category = classify_email(email_text, categories_weights)
    print(f'{file_name} -> {category}')

  else:
    print(f'{file_name} -> Ошибки')

mail_0001.txt -> Черновик
mail_0002.txt -> Важное
mail_0003.txt -> Информация
mail_0004.txt -> Важное
mail_0005.txt -> Черновик
mail_0006.txt -> Счета
mail_0007.txt -> Черновик
mail_0008.txt -> Информация
mail_0009.txt -> Черновик
mail_0010.txt -> Важное
mail_0011.txt -> HR
mail_0012.txt -> Черновик
mail_0013.txt -> Счета
mail_0014.txt -> Технические ошибки
mail_0015.txt -> HR
mail_0016.txt -> Технические ошибки
mail_0017.txt -> Счета
mail_0018.txt -> Технические ошибки
mail_0019.txt -> Документы
mail_0020.txt -> Черновик
mail_0021.txt -> Технические ошибки
mail_0022.txt -> Черновик
mail_0023.txt -> Черновик
mail_0024.txt -> Подтверждение доступа к аккаунту
mail_0025.txt -> HR
mail_0026.txt -> Не определено
mail_0027.txt -> HR
mail_0028.txt -> Технические ошибки
mail_0029.txt -> Информация
mail_0030.txt -> Подтверждение доступа к аккаунту
mail_0031.txt -> Важное
mail_0032.txt -> Счета
mail_0033.txt -> Черновик
mail_0034.txt -> Счета
mail_0035.txt -> Черновик
mail_0036.txt -> Информация